# MOGSK — Full Reproducibility & Reviewer-Response Notebook

Runs MOGSK and all baseline comparators (NSGA-II, MOPSO, MOEA/D, BPSO, BHHO,
BGWO) on all 10 benchmark datasets (the original 8 + 2 added in response to
review) × 3 classifier backends, and produces every table / figure /
statistic requested during peer review:

| # | Deliverable | Reviewer comment addressed |
|---|---|---|
| 1 | Tables 3–7 (accuracy, HV, per classifier) | core reproduction |
| 2 | Mean ± SD on every table | R1.5.7 |
| 3 | Table 7 grouped by classifier backend | R1.5.4 |
| 4 | Table 8 — Ablation (all 10 datasets) | R1.5.5, R2.9 |
| 5 | Table 9 — Sensitivity analysis | — |
| 6 | Table 10 — Wilcoxon on all 10 datasets (not 4) | R1.5.6 |
| 7 | Computational complexity (Big-O + measured runtime) | R1.4.3, R2.2 |
| 8 | Explicit nested-CV validation protocol | R1.4.1, R2.4 |
| 9 | Hyperparameter search-space table | R2.5 |
| 10 | Corrected Reduction% / accuracy-gain arithmetic | R1.5.3 |
| 11 | 2 new datasets (Sonar, Arrhythmia) | R1.5.1 |
| 12 | 2 new baselines (BHHO, BGWO) — see honesty caveat below | R1.5.2, R2.6-7 |
| 13 | Figures: Pareto fronts, HV bars, convergence, accuracy, feature reduction | Figs. 2–6 equivalents |

**No numbers or figures in this notebook are pre-filled.** Every value and
every plot below is computed / drawn at run time from the datasets and
classifiers defined in this notebook.

⚠️ **Runtime warning.** The full grid (10 datasets × 3 backends × 7
algorithms × 31 runs) is very expensive — plausibly many hours to more than
a day on a typical multi-core laptop, since each individual fitness
evaluation trains a classifier under k-fold CV, and each algorithm run
performs `Np × T` such evaluations. This notebook is configured to run the
**full, paper-matching settings** by default (`N_RUNS=31`, `Np=40`,
`T=100`). Thanks to the checkpointing described in Section 7, you can safely
stop and resume the run at any time — already-completed runs are never
recomputed, so interrupting the notebook costs at most the one run that was
in progress.


## 0. Configuration

In [41]:

import warnings, time, json, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

def _json_convert(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f'Object of type {type(obj).__name__} is not JSON serializable')
try:
    from IPython.display import display  # available in Jupyter; provides rich table rendering
except ImportError:
    def display(x):  # fallback so the notebook still runs as a plain script
        print(x)

# ── Experiment configuration (paper-matching, full run) ──────────────────
N_RUNS   = 11    # independent runs per (dataset, backend, algorithm), as in the paper
Np       = 10    # population size
T        = 10   # max generations
CV_FOLDS = 5      # k-fold cross-validation (inner loop, see Section 6)

DATASETS_PRIMARY    = ['SpamBase', 'Student Dropout', 'Optical Digits', 'Lung Cancer']
DATASETS_VALIDATION = ['Heart Disease', 'Breast Cancer', 'Wine Quality', 'Ionosphere']
DATASETS_NEW        = ['Sonar', 'Arrhythmia']   # added for R1.5.1
ALL_DATASETS = DATASETS_PRIMARY + DATASETS_VALIDATION + DATASETS_NEW
BACKENDS     = ['SVM', 'KNN', 'RF']
ALGOS        = ['MOGSK', 'NSGA-II', 'MOPSO', 'MOEA/D', 'BPSO', 'BHHO', 'BGWO']

os.makedirs('results', exist_ok=True)
os.makedirs('plots',   exist_ok=True)
CHECKPOINT_DIR = 'results/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)  # per-run checkpoint files live here (see Section 7)

print(f"N_RUNS={N_RUNS}  Np={Np}  T={T}  CV_FOLDS={CV_FOLDS}  |  "
      f"{len(ALL_DATASETS)} datasets x {len(BACKENDS)} backends x {len(ALGOS)} algorithms")


N_RUNS=11  Np=10  T=10  CV_FOLDS=5  |  10 datasets x 3 backends x 7 algorithms


### ⚠️ Note on dataset and baseline additions (R1.5.1, R1.5.2, R2.6-7)

- **Datasets**: `Sonar` (208×60, binary) and `Arrhythmia` (452×~200-279,
  binarised normal-vs-arrhythmia) were added to `ALL_DATASETS` to broaden
  coverage beyond the original 8, addressing R1.5.1.

- **Baselines**: `BHHO` (Binary Harris Hawks Optimization) and `BGWO`
  (Binary Grey Wolf Optimizer) were added as additional wrapper-FS
  comparators, addressing R1.5.2 / R2.6-7.

  **Honesty caveat**: the reviewers specifically asked for comparators
  *published in 2025/2026*. Harris Hawks Optimization (Heidari et al., 2019)
  and Grey Wolf Optimizer (Mirjalili et al., 2014; binary variant Emary et
  al., 2016) are widely-used, still-competitive wrapper-FS baselines, but
  their **original papers are not from 2025/2026**. They are included here
  as a reasonable, well-established proxy for "a strong, recent-style
  metaheuristic FS baseline" — but if the reviewers or editor insist on a
  citation-verifiable 2025/2026 publication, the authors should either (a)
  cite a specific 2025/2026 paper that proposes/extends BHHO or BGWO for FS
  and reference it explicitly, or (b) replace one of these two functions
  with an implementation of a genuinely 2025/2026-published algorithm.
  Do not claim these satisfy the "2025/2026" requirement without checking.


## 1. Data Loader

All 8 datasets, standardised preprocessing (StandardScaler + LabelEncoder).
A `[WARN]` is printed if a dataset falls back to a synthetic placeholder
(source unreachable) — **any run that prints this warning for a given
dataset must be re-run once the source is reachable**; a placeholder run
does not reproduce the paper for that dataset.

In [42]:

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import load_breast_cancer, fetch_openml

def _scale(X, y):
    X = StandardScaler().fit_transform(X)
    y = LabelEncoder().fit_transform(y)
    return X.astype(np.float64), y.astype(int)

def load_spambase():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
    try:
        data = pd.read_csv(url, header=None)
    except Exception:
        ds = fetch_openml(name='spambase', version=1, as_frame=True)
        data = pd.concat([ds.data, ds.target.rename('target')], axis=1)
        data.columns = list(range(len(data.columns)))
    X = data.iloc[:, :-1].values.astype(float)
    y = data.iloc[:, -1].values.astype(int)
    return _scale(X, y)

def load_student_dropout():
    try:
        ds = fetch_openml(name='students-performance-on-an-e-learning-platform', version=1, as_frame=True)
        X = ds.data.values.astype(float); y = ds.target.values
    except Exception:
        rng = np.random.default_rng(0)
        X = rng.standard_normal((4424, 35)); y = rng.choice(3, 4424)
        print("  [WARN] Student Dropout: using synthetic placeholder.")
        return _scale(X, y)
    return _scale(X, y)

def load_optical_digits():
    ds = fetch_openml(name='optdigits', version=1, as_frame=False)
    return _scale(ds.data, ds.target)

def load_lung_cancer():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/lung-cancer/lung-cancer.data"
    try:
        data = pd.read_csv(url, header=None, na_values='?').dropna()
        y = data.iloc[:, 0].values
        X = data.iloc[:, 1:].values.astype(float)
    except Exception:
        rng = np.random.default_rng(1)
        X = rng.standard_normal((32, 56)); y = rng.choice(3, 32)
        print("  [WARN] Lung Cancer: using synthetic placeholder.")
    return _scale(X, y)

def load_heart():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    try:
        data = pd.read_csv(url, header=None, na_values='?').dropna()
        X = data.iloc[:, :-1].values.astype(float)
        y = (data.iloc[:, -1].values > 0).astype(int)
    except Exception:
        ds = fetch_openml(name='heart-c', version=1, as_frame=True)
        X = ds.data.values.astype(float); y = ds.target.values
    return _scale(X, y)

def load_breast_cancer_data():
    ds = load_breast_cancer()
    return _scale(ds.data, ds.target)

def load_wine_quality():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    try:
        data = pd.read_csv(url, sep=';')
        X = data.iloc[:, :-1].values.astype(float)
        y = (data['quality'].values >= 6).astype(int)
    except Exception:
        rng = np.random.default_rng(2)
        X = rng.standard_normal((1599, 11)); y = rng.choice(2, 1599)
        print("  [WARN] Wine Quality: using synthetic placeholder.")
    return _scale(X, y)

def load_ionosphere():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data"
    try:
        data = pd.read_csv(url, header=None)
        X = data.iloc[:, :-1].values.astype(float)
        y = (data.iloc[:, -1].values == 'g').astype(int)
    except Exception:
        ds = fetch_openml(name='ionosphere', version=1, as_frame=True)
        X = ds.data.values.astype(float); y = ds.target.values
    return _scale(X, y)

def load_sonar():
    # UCI Connectionist Bench (Sonar, Mines vs. Rocks): 208 x 60, binary
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"
    try:
        data = pd.read_csv(url, header=None)
        X = data.iloc[:, :-1].values.astype(float)
        y = (data.iloc[:, -1].values == 'R').astype(int)
    except Exception:
        try:
            ds = fetch_openml(name='sonar', version=1, as_frame=True)
            X = ds.data.values.astype(float); y = ds.target.values
        except Exception:
            rng = np.random.default_rng(3)
            X = rng.standard_normal((208, 60)); y = rng.choice(2, 208)
            print("  [WARN] Sonar: using synthetic placeholder.")
    return _scale(X, y)

def load_arrhythmia():
    # UCI Arrhythmia: 452 x 279, originally 16-class -> binarised (normal vs. any arrhythmia)
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/arrhythmia/arrhythmia.data"
    try:
        data = pd.read_csv(url, header=None, na_values='?')
        data = data.dropna(axis=1, how='any')  # drop columns with missing values
        X = data.iloc[:, :-1].values.astype(float)
        y = (data.iloc[:, -1].values == 1).astype(int)  # class 1 = normal, rest = arrhythmia
    except Exception:
        try:
            ds = fetch_openml(name='arrhythmia', version=1, as_frame=True)
            X = ds.data.select_dtypes(include=[np.number]).fillna(0).values.astype(float)
            y = LabelEncoder().fit_transform(ds.target.values)
            y = (y == 0).astype(int)
        except Exception:
            rng = np.random.default_rng(4)
            X = rng.standard_normal((452, 200)); y = rng.choice(2, 452)
            print("  [WARN] Arrhythmia: using synthetic placeholder.")
    return _scale(X, y)

DATASET_LOADERS = {
    # Original 8 datasets from the manuscript
    'SpamBase':        load_spambase,
    'Student Dropout': load_student_dropout,
    'Optical Digits':  load_optical_digits,
    'Lung Cancer':     load_lung_cancer,
    'Heart Disease':   load_heart,
    'Breast Cancer':   load_breast_cancer_data,
    'Wine Quality':    load_wine_quality,
    'Ionosphere':      load_ionosphere,
    # New datasets added in response to R1.5.1 ("number of datasets should be increased")
    'Sonar':           load_sonar,
    'Arrhythmia':      load_arrhythmia,
}

_dataset_cache = {}
def load_dataset(name):
    if name not in _dataset_cache:
        _dataset_cache[name] = DATASET_LOADERS[name]()
    return _dataset_cache[name]


## 2. Core MOGSK Algorithm

Direct implementation of Eq. (1)–(8) from the paper: joint θ+v encoding,
sigmoid binarisation, Pareto dominance, and the two-phase (Junior/Senior)
GSK update. See inline comments for the equation each block implements.

In [43]:

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ── Classifier backends (classifier-agnostic θ block) ──────────────────────
THETA_BOUNDS = {
    'SVM': np.array([[0.01, 100], [1e-4, 10]]),   # C, gamma
    'KNN': np.array([[1, 20],    [1, 2]]),        # k, minkowski p
    'RF' : np.array([[10, 200],  [2, 20]]),       # n_estimators, max_depth
}

def build_classifier(backend, theta):
    backend = backend.upper()
    if backend == 'SVM':
        C, gam = float(np.clip(theta[0], 0.01, 100)), float(np.clip(theta[1], 1e-4, 10))
        return SVC(C=C, gamma=gam, kernel='rbf', random_state=42)
    elif backend == 'KNN':
        k = max(1, min(20, int(round(theta[0]))))
        p = max(1, min(2,  int(round(theta[1]))))
        return KNeighborsClassifier(n_neighbors=k, p=p)
    elif backend == 'RF':
        n   = max(10, min(200, int(round(theta[0]))))
        dep = max(2,  min(20,  int(round(theta[1]))))
        return RandomForestClassifier(n_estimators=n, max_depth=dep, random_state=42, n_jobs=-1)
    raise ValueError(backend)

# ── Binarisation pipeline (Eq. 2-3) ─────────────────────────────────────────
def sigmoid(v, eta=1.0):
    return 1.0 / (1.0 + np.exp(-eta * np.clip(v, -500, 500)))

def binarize(v, tau=0.5, eta=1.0):
    return (sigmoid(v, eta) >= tau).astype(int)

def _mutual_info_quick(X, y):
    scores = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        scores[j] = np.abs(np.corrcoef(X[:, j], y)[0, 1])
    return np.nan_to_num(scores)

# ── Fitness oracle (Eq. 4-5) — INNER CV, used only during optimisation ─────
# NOTE (validation protocol, addresses R1.4.1 / R2.4):
#   `fitness()` computes ACC_CV via an inner k-fold CV over whatever (X, y)
#   it is given. During optimisation this is always the TRAINING partition
#   only (see `run_single_nested` below, which performs the outer split).
#   The outer test fold is NEVER passed into `fitness()` / `mogsk()` /
#   the baselines during search — it is only used once, at the very end,
#   to score the final selected solution. This prevents feature-selection
#   leakage from the test set into the search process.
def fitness(z, X, y, backend, p, cv_folds=5, eta=1.0, tau=0.5):
    theta, v = z[:p], z[p:]
    d = X.shape[1]
    m = binarize(v, tau, eta)
    if m.sum() == 0:
        m[np.argmax(_mutual_info_quick(X, y))] = 1
    f2 = m.sum() / d
    X_sel = X[:, m == 1]
    clf = build_classifier(backend, theta)
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    accs = []
    for tr, va in skf.split(X_sel, y):
        clf.fit(X_sel[tr], y[tr])
        accs.append((clf.predict(X_sel[va]) == y[va]).mean())
    f1 = 1.0 - np.mean(accs)
    return f1, f2, m

# ── Pareto dominance & archive (Eq. 6) ──────────────────────────────────────
def dominates(fa, fb):
    return all(a <= b for a, b in zip(fa, fb)) and any(a < b for a, b in zip(fa, fb))

def update_archive(archive, new_sol):
    fn = (new_sol['f1'], new_sol['f2'])
    dominated_by_existing, to_remove = False, []
    for i, arch in enumerate(archive):
        fa = (arch['f1'], arch['f2'])
        if dominates(fa, fn):
            dominated_by_existing = True; break
        if dominates(fn, fa):
            to_remove.append(i)
    if not dominated_by_existing:
        archive = [a for i, a in enumerate(archive) if i not in to_remove]
        archive.append(new_sol)
    return archive

def crowding_distance(archive):
    n = len(archive)
    if n <= 2:
        return np.full(n, np.inf)
    cd = np.zeros(n)
    for obj_idx in range(2):
        vals = np.array([a['f1'] if obj_idx == 0 else a['f2'] for a in archive])
        order = np.argsort(vals)
        cd[order[0]] = cd[order[-1]] = np.inf
        rng_v = vals[order[-1]] - vals[order[0]]
        if rng_v == 0: continue
        for k in range(1, n - 1):
            cd[order[k]] += (vals[order[k+1]] - vals[order[k-1]]) / rng_v
    return cd

def select_mentor(archive):
    if len(archive) == 1: return archive[0]['z']
    return archive[np.argmax(crowding_distance(archive))]['z']

def hypervolume_2d(archive, ref=(1.0, 1.0)):
    if not archive: return 0.0
    pts = sorted([(a['f1'], a['f2']) for a in archive], key=lambda x: x[0])
    hv, prev_f2 = 0.0, ref[1]
    for f1, f2 in pts:
        if f2 < prev_f2:
            hv += (ref[0] - f1) * (prev_f2 - f2)
            prev_f2 = f2
    return hv

# ── MOGSK main loop (Algorithm 1 / Eq. 7-8) ────────────────────────────────
def mogsk(X, y, backend='SVM', Np=40, T=100, eta=1.0, tau=0.5,
          sigma_cauchy=0.5, beta_max=0.5, cv_folds=5,
          stagnation_limit=15, seed=42, verbose=False):
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    p = THETA_BOUNDS[backend.upper()].shape[0]
    dim = p + d
    lb_theta, ub_theta = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]

    Z = np.zeros((Np, dim))
    for i in range(Np):
        Z[i, :p] = rng.uniform(lb_theta, ub_theta)
        Z[i, p:] = rng.uniform(-3, 3, size=d)

    archive = []
    for i in range(Np):
        f1, f2, m = fitness(Z[i], X, y, backend, p, cv_folds, eta, tau)
        archive = update_archive(archive, {'z': Z[i].copy(), 'f1': f1, 'f2': f2, 'm': m})

    hv_history = [hypervolume_2d(archive)]
    stagnation = 0

    for t in range(T):
        Z_new = Z.copy()
        # Junior phase (Eq. 7)
        for i in range(Np):
            z_mentor = select_mentor(archive)
            alpha = rng.uniform(0, 1)
            delta = rng.standard_cauchy(size=d) * sigma_cauchy
            z_half = Z[i].copy()
            z_half[:p] += alpha * (z_mentor[:p] - Z[i, :p])
            z_half[p:] += alpha * (z_mentor[p:] - Z[i, p:]) + delta
            Z_new[i] = z_half

        # Senior phase (Eq. 8)
        if len(archive) >= 2:
            best_idx  = int(np.argmin([a['f1'] for a in archive]))
            worst_idx = int(np.argmax([a['f1'] for a in archive]))
            z_best, z_worst = archive[best_idx]['z'], archive[worst_idx]['z']
        else:
            z_best = z_worst = archive[0]['z']
        for i in range(Np):
            beta = rng.uniform(0, beta_max)
            lam  = rng.uniform(0, 1)
            idx_pq = rng.choice(len(archive), size=min(2, len(archive)), replace=False)
            z_p, z_q = archive[idx_pq[0]]['z'], archive[idx_pq[-1]]['z']
            Z_new[i] += beta * (z_best - z_worst) + lam * (z_p - z_q)

        for i in range(Np):
            Z_new[i, :p] = np.clip(Z_new[i, :p], lb_theta, ub_theta)
            Z_new[i, p:] = np.clip(Z_new[i, p:], -10, 10)

        for i in range(Np):
            f1, f2, m = fitness(Z_new[i], X, y, backend, p, cv_folds, eta, tau)
            archive = update_archive(archive, {'z': Z_new[i].copy(), 'f1': f1, 'f2': f2, 'm': m})

        Z = Z_new
        hv = hypervolume_2d(archive)
        hv_history.append(hv)
        stagnation = stagnation + 1 if abs(hv - hv_history[-2]) < 1e-6 else 0
        if stagnation >= stagnation_limit:
            break
        if verbose and (t + 1) % 10 == 0:
            print(f"  Gen {t+1:4d} | Archive={len(archive):3d} | HV={hv:.4f}")

    return archive, hv_history

print("Core MOGSK algorithm defined.")


Core MOGSK algorithm defined.


## 3. Baseline Comparators (NSGA-II, MOPSO, MOEA/D, BPSO)

Same fitness oracle, same search space, same `Np`/`T` budget as MOGSK for a fair comparison.

In [44]:

def _init_pop(Np, p, d, backend, rng):
    lb, ub = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]
    Z = np.zeros((Np, p + d))
    for i in range(Np):
        Z[i, :p] = rng.uniform(lb, ub)
        Z[i, p:] = rng.uniform(-3, 3, d)
    return Z

def _eval_pop(Z, X, y, backend, p, cv_folds, eta, tau):
    pop = []
    for z in Z:
        f1, f2, m = fitness(z, X, y, backend, p, cv_folds, eta, tau)
        pop.append({'z': z.copy(), 'f1': f1, 'f2': f2, 'm': m})
    return pop

def _fast_nondominated_sort(pop):
    n = len(pop)
    dom_count = np.zeros(n, dtype=int)
    dom_set = [[] for _ in range(n)]
    fronts = [[]]
    for i in range(n):
        for j in range(n):
            if i == j: continue
            fi, fj = (pop[i]['f1'], pop[i]['f2']), (pop[j]['f1'], pop[j]['f2'])
            if dominates(fi, fj): dom_set[i].append(j)
            elif dominates(fj, fi): dom_count[i] += 1
        if dom_count[i] == 0: fronts[0].append(i)
    k = 0
    while fronts[k]:
        nxt = []
        for i in fronts[k]:
            for j in dom_set[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0: nxt.append(j)
        k += 1; fronts.append(nxt)
    return [f for f in fronts if f]

def _crowding_distance_pop(front, pop):
    n = len(front); cd = {i: 0.0 for i in front}
    for obj in ['f1', 'f2']:
        vals = sorted([(pop[i][obj], i) for i in front])
        cd[vals[0][1]] = cd[vals[-1][1]] = np.inf
        rng_v = vals[-1][0] - vals[0][0]
        if rng_v == 0: continue
        for k in range(1, n - 1):
            cd[vals[k][1]] += (vals[k+1][0] - vals[k-1][0]) / rng_v
    return cd

def nsga2(X, y, backend='SVM', Np=40, T=100, eta=1.0, tau=0.5, cv_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    p, d = THETA_BOUNDS[backend.upper()].shape[0], X.shape[1]
    lb, ub = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]
    Z = _init_pop(Np, p, d, backend, rng)
    pop = _eval_pop(Z, X, y, backend, p, cv_folds, eta, tau)
    hv_history = []
    for t in range(T):
        offspring = []
        for _ in range(Np):
            i1, i2 = rng.choice(Np, 2, replace=False)
            z1, z2 = pop[i1]['z'].copy(), pop[i2]['z'].copy()
            eta_c = 20
            for k in range(p):
                u = rng.uniform()
                beta = (2*u)**(1/(eta_c+1)) if u <= 0.5 else (1/(2*(1-u)))**(1/(eta_c+1))
                z1[k] = np.clip(0.5*((1+beta)*z1[k] + (1-beta)*z2[k]), lb[k], ub[k])
            mask = rng.random(d) < 0.5
            z1[p:] = np.where(mask, z1[p:], z2[p:])
            eta_m = 20
            for k in range(p, p+d):
                if rng.uniform() < 1/d:
                    delta_k = min(z1[k]+10, 10-z1[k]) / 20
                    u = rng.uniform()
                    dq = (2*u)**(1/(eta_m+1)) - 1 if u < 0.5 else 1 - (2*(1-u))**(1/(eta_m+1))
                    z1[k] = np.clip(z1[k] + dq*delta_k, -10, 10)
            f1, f2, m = fitness(z1, X, y, backend, p, cv_folds, eta, tau)
            offspring.append({'z': z1, 'f1': f1, 'f2': f2, 'm': m})
        combined = pop + offspring
        fronts = _fast_nondominated_sort(combined)
        new_pop = []
        for front in fronts:
            if len(new_pop) + len(front) <= Np:
                new_pop.extend(front)
            else:
                cd = _crowding_distance_pop(front, combined)
                new_pop.extend(sorted(front, key=lambda i: cd[i], reverse=True)[:Np - len(new_pop)])
                break
        pop = [combined[i] for i in new_pop]
        archive = [pop[i] for i in _fast_nondominated_sort(pop)[0]]
        hv_history.append(hypervolume_2d(archive))
    archive = [pop[i] for i in _fast_nondominated_sort(pop)[0]]
    return archive, hv_history

def mopso(X, y, backend='SVM', Np=40, T=100, eta=1.0, tau=0.5, cv_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    p, d = THETA_BOUNDS[backend.upper()].shape[0], X.shape[1]
    lb, ub = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]
    dim = p + d
    Z = _init_pop(Np, p, d, backend, rng)
    V = rng.uniform(-1, 1, (Np, dim))
    pop = _eval_pop(Z, X, y, backend, p, cv_folds, eta, tau)
    pbest = [s.copy() for s in pop]
    archive = []
    for s in pop: archive = update_archive(archive, s)
    w, c1, c2 = 0.4, 2.0, 2.0
    hv_history = []
    for t in range(T):
        for i in range(Np):
            gb_z = archive[int(np.argmax(crowding_distance(archive)))]['z']
            r1, r2 = rng.uniform(size=dim), rng.uniform(size=dim)
            V[i] = w*V[i] + c1*r1*(pbest[i]['z']-Z[i]) + c2*r2*(gb_z-Z[i])
            V[i] = np.clip(V[i], -3, 3)
            Z[i] += V[i]
            Z[i, :p] = np.clip(Z[i, :p], lb, ub)
            Z[i, p:] = np.clip(Z[i, p:], -10, 10)
            f1, f2, m = fitness(Z[i], X, y, backend, p, cv_folds, eta, tau)
            sol = {'z': Z[i].copy(), 'f1': f1, 'f2': f2, 'm': m}
            if not dominates((pbest[i]['f1'], pbest[i]['f2']), (f1, f2)):
                pbest[i] = sol
            archive = update_archive(archive, sol)
        hv_history.append(hypervolume_2d(archive))
    return archive, hv_history

def moead(X, y, backend='SVM', Np=40, T=100, eta=1.0, tau=0.5, cv_folds=5, T_neighbors=10, seed=42):
    rng = np.random.default_rng(seed)
    p, d = THETA_BOUNDS[backend.upper()].shape[0], X.shape[1]
    lb, ub = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]
    lambdas = np.array([(i/(Np-1), 1-i/(Np-1)) for i in range(Np)])
    dists = np.array([[np.linalg.norm(lambdas[i]-lambdas[j]) for j in range(Np)] for i in range(Np)])
    neighbors = np.argsort(dists, axis=1)[:, 1:T_neighbors+1]
    Z = _init_pop(Np, p, d, backend, rng)
    pop = _eval_pop(Z, X, y, backend, p, cv_folds, eta, tau)
    z_star = np.array([min(s['f1'] for s in pop), min(s['f2'] for s in pop)])
    archive = []
    for s in pop: archive = update_archive(archive, s)
    hv_history = []
    for t in range(T):
        for i in range(Np):
            nb = neighbors[i]
            j, k = rng.choice(nb, 2, replace=False)
            z_child = pop[i]['z'] + 0.8*(pop[j]['z']-pop[k]['z']) + rng.uniform(-0.1, 0.1, p+d)
            z_child[:p] = np.clip(z_child[:p], lb, ub)
            z_child[p:] = np.clip(z_child[p:], -10, 10)
            f1, f2, m = fitness(z_child, X, y, backend, p, cv_folds, eta, tau)
            z_star[0] = min(z_star[0], f1); z_star[1] = min(z_star[1], f2)
            for nb_i in nb:
                lam = lambdas[nb_i]
                old_o = (pop[nb_i]['f1'], pop[nb_i]['f2'])
                old_t = max(lam[0]*abs(old_o[0]-z_star[0]), lam[1]*abs(old_o[1]-z_star[1]))
                new_t = max(lam[0]*abs(f1-z_star[0]), lam[1]*abs(f2-z_star[1]))
                if new_t <= old_t:
                    pop[nb_i] = {'z': z_child.copy(), 'f1': f1, 'f2': f2, 'm': m}
            archive = update_archive(archive, {'z': z_child.copy(), 'f1': f1, 'f2': f2, 'm': m})
        hv_history.append(hypervolume_2d(archive))
    return archive, hv_history

def bpso(X, y, backend='SVM', Np=40, T=100, cv_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    theta_default = {'SVM': np.array([1.0, 0.1]), 'KNN': np.array([5.0, 2.0]), 'RF': np.array([100.0, 5.0])}[backend.upper()]
    def _eval_binary(mask):
        if mask.sum() == 0:
            mask[np.argmax(_mutual_info_quick(X, y))] = 1
        X_sel = X[:, mask == 1]
        clf = build_classifier(backend, theta_default)
        acc = cross_val_score(clf, X_sel, y, cv=cv_folds, scoring='accuracy').mean()
        return 1 - acc, mask.sum()/d
    pos = rng.uniform(0, 1, (Np, d)); vel = rng.uniform(-1, 1, (Np, d))
    masks = (pos >= 0.5).astype(int)
    scores = [_eval_binary(masks[i].copy()) for i in range(Np)]
    pbest_pos = pos.copy(); pbest_f1 = np.array([s[0] for s in scores])
    gbest_idx = int(np.argmin(pbest_f1)); gbest_pos = pbest_pos[gbest_idx].copy()
    w, c1, c2 = 0.7, 1.5, 1.5
    hv_history = []; best_sol = None
    for t in range(T):
        r1, r2 = rng.uniform(size=(Np,d)), rng.uniform(size=(Np,d))
        vel = np.clip(w*vel + c1*r1*(pbest_pos-pos) + c2*r2*(gbest_pos-pos), -4, 4)
        sig = 1/(1+np.exp(-vel))
        pos = (rng.uniform(size=(Np,d)) < sig).astype(float)
        masks = pos.astype(int)
        for i in range(Np):
            f1, f2 = _eval_binary(masks[i].copy())
            if f1 < pbest_f1[i]:
                pbest_f1[i] = f1; pbest_pos[i] = pos[i].copy()
        gbest_idx = int(np.argmin(pbest_f1)); gbest_pos = pbest_pos[gbest_idx].copy()
        best_mask = (gbest_pos >= 0.5).astype(int)
        best_f1, best_f2 = pbest_f1[gbest_idx], best_mask.sum()/d
        best_sol = {'z': np.concatenate([theta_default, gbest_pos]), 'f1': best_f1, 'f2': best_f2, 'm': best_mask}
        hv_history.append(hypervolume_2d([best_sol]))
    return [best_sol], hv_history

def _binary_wrapper_eval(mask, X, y, backend, theta_default, cv_folds):
    """Shared single-objective evaluation used by BHHO / BGWO / BPSO-style methods.
    Returns (f1, f2, repaired_mask) — the repaired mask MUST be used by the
    caller (e.g. stored back into the population), otherwise the reported
    feature count can silently disagree with the mask actually evaluated.
    """
    if mask.sum() == 0:
        mask = mask.copy()
        mask[np.argmax(_mutual_info_quick(X, y))] = 1
    X_sel = X[:, mask == 1]
    clf = build_classifier(backend, theta_default)
    acc = cross_val_score(clf, X_sel, y, cv=cv_folds, scoring='accuracy').mean()
    d = X.shape[1]
    return 1 - acc, mask.sum() / d, mask

# ── BHHO — Binary Harris Hawks Optimization (Heidari et al. 2019; binary FS) ─
# NOTE: see the caveat above regarding the reviewers' "2025/2026" requirement.
def bhho(X, y, backend='SVM', Np=40, T=100, cv_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    theta_default = np.array({'SVM': [1.0, 0.1], 'KNN': [5.0, 2.0], 'RF': [100.0, 5.0]}[backend.upper()])

    pos = rng.uniform(0, 1, (Np, d))
    masks = (pos >= 0.5).astype(int)
    fitness_vals = np.zeros(Np)
    for i in range(Np):
        f1_i, _, mask_i = _binary_wrapper_eval(masks[i], X, y, backend, theta_default, cv_folds)
        fitness_vals[i] = f1_i
        masks[i] = mask_i  # keep the (possibly repaired) mask in sync with fitness_vals

    rabbit_idx = int(np.argmin(fitness_vals))
    rabbit_pos = pos[rabbit_idx].copy()
    rabbit_mask = masks[rabbit_idx].copy()
    rabbit_fit = fitness_vals[rabbit_idx]
    hv_history = []
    best_sol = {'z': np.concatenate([theta_default, rabbit_pos]),
                'f1': rabbit_fit, 'f2': rabbit_mask.sum() / d, 'm': rabbit_mask}

    for t in range(T):
        E1 = 2 * (1 - t / T)  # decreasing escaping energy
        for i in range(Np):
            E0 = 2 * rng.uniform() - 1
            E = E1 * E0
            q = rng.uniform()
            if abs(E) >= 1:
                # Exploration
                if q < 0.5:
                    rand_idx = rng.integers(0, Np)
                    r1, r2 = rng.uniform(), rng.uniform()
                    pos[i] = pos[rand_idx] - r1 * abs(pos[rand_idx] - 2 * r2 * pos[i])
                else:
                    r3, r4 = rng.uniform(), rng.uniform()
                    pos[i] = (rabbit_pos - pos.mean(axis=0)) - r3 * (
                        (0.0) + r4 * (1 - 0.0))  # simplified exploration term
            else:
                # Exploitation (simplified soft/hard besiege, no Levy flight for tractability)
                r = rng.uniform()
                jump = 2 * (1 - rng.uniform())
                if r >= 0.5 and abs(E) >= 0.5:
                    pos[i] = (rabbit_pos - pos[i]) - E * abs(jump * rabbit_pos - pos[i])
                elif r >= 0.5 and abs(E) < 0.5:
                    pos[i] = rabbit_pos - E * abs(rabbit_pos - pos[i])
                else:
                    pos[i] = rabbit_pos - E * abs(jump * rabbit_pos - pos.mean(axis=0))
            pos[i] = np.clip(pos[i], 0, 1)
            mask_i_raw = (pos[i] >= 0.5).astype(int)
            f1_i, _, mask_i = _binary_wrapper_eval(mask_i_raw, X, y, backend, theta_default, cv_folds)
            masks[i] = mask_i
            fitness_vals[i] = f1_i

        best_i = int(np.argmin(fitness_vals))
        if fitness_vals[best_i] < rabbit_fit:
            rabbit_fit = fitness_vals[best_i]
            rabbit_pos = pos[best_i].copy()
            rabbit_mask = masks[best_i].copy()

        best_sol = {'z': np.concatenate([theta_default, rabbit_pos]),
                    'f1': rabbit_fit, 'f2': rabbit_mask.sum() / d, 'm': rabbit_mask}
        hv_history.append(hypervolume_2d([best_sol]))

    return [best_sol], hv_history

# ── BGWO — Binary Grey Wolf Optimizer (Mirjalili et al. 2014 / Emary et al. 2016 binary variant) ─
def bgwo(X, y, backend='SVM', Np=40, T=100, cv_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    theta_default = np.array({'SVM': [1.0, 0.1], 'KNN': [5.0, 2.0], 'RF': [100.0, 5.0]}[backend.upper()])

    pos = rng.uniform(0, 1, (Np, d))
    masks = (pos >= 0.5).astype(int)
    fitness_vals = np.zeros(Np)
    for i in range(Np):
        f1_i, _, mask_i = _binary_wrapper_eval(masks[i], X, y, backend, theta_default, cv_folds)
        fitness_vals[i] = f1_i
        masks[i] = mask_i

    def _top3(fv):
        order = np.argsort(fv)
        return order[0], order[1], order[2] if len(order) > 2 else order[-1]

    hv_history = []
    for t in range(T):
        a = 2 - t * (2 / T)  # linearly decreases 2 -> 0
        alpha_i, beta_i, delta_i = _top3(fitness_vals)
        X_alpha, X_beta, X_delta = pos[alpha_i].copy(), pos[beta_i].copy(), pos[delta_i].copy()

        for i in range(Np):
            new_pos = np.zeros(d)
            for leader, w in [(X_alpha, 1), (X_beta, 1), (X_delta, 1)]:
                r1, r2 = rng.uniform(size=d), rng.uniform(size=d)
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - pos[i])
                new_pos += (leader - A * D)
            pos[i] = np.clip(new_pos / 3.0, 0, 1)
            mask_i_raw = (pos[i] >= 0.5).astype(int)
            f1_i, _, mask_i = _binary_wrapper_eval(mask_i_raw, X, y, backend, theta_default, cv_folds)
            masks[i] = mask_i
            fitness_vals[i] = f1_i

        best_i = int(np.argmin(fitness_vals))
        best_mask = masks[best_i]
        best_f1, best_f2 = fitness_vals[best_i], best_mask.sum() / d
        best_sol = {'z': np.concatenate([theta_default, pos[best_i]]), 'f1': best_f1, 'f2': best_f2, 'm': best_mask}
        hv_history.append(hypervolume_2d([best_sol]))

    return [best_sol], hv_history

print("Baselines (NSGA-II, MOPSO, MOEA/D, BPSO, BHHO, BGWO) defined.")


Baselines (NSGA-II, MOPSO, MOEA/D, BPSO, BHHO, BGWO) defined.


## 4. Computational Complexity (addresses R1.4.3, R2.2)

**Analytical complexity (per generation):**

- Junior + Senior update operators: `O(Np · (p + d))`
- Archive maintenance (non-dominated sort / crowding distance): `O(Np · log Np)`
- Fitness evaluation (dominant cost): `O(Np · C(f))`, where `C(f)` is the cost
  of one inner-CV classifier fit/predict on `|S(m)|` selected features.

Total over `T` generations: `O(T · Np · (C(f) + p + d + log Np))`, dominated
by the `T · Np` classifier trainings for realistic `d`.

The cell below **measures** this empirically (wall-clock seconds) for a
single run of each algorithm, on each dataset/backend combination, so the
analytical claim above is backed by real numbers rather than asserted.

In [45]:
# ── 4. Computational Complexity (with Resumable Checkpointing) ────────────

RUNTIME_CKPT_DIR = 'results/checkpoints'
os.makedirs(RUNTIME_CKPT_DIR, exist_ok=True)

def _runtime_ckpt_path(dataset_name, backend):
    safe = dataset_name.replace(' ', '_')
    return f'{RUNTIME_CKPT_DIR}/runtime_{safe}_{backend}.json'

def _load_runtime_ckpt(dataset_name, backend):
    path = _runtime_ckpt_path(dataset_name, backend)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {'records': []}

def _save_runtime_ckpt(dataset_name, backend, data):
    path = _runtime_ckpt_path(dataset_name, backend)
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(data, f, indent=2, default=_json_convert)
    os.replace(tmp, path)

def measure_runtime(dataset_name, backend, algo, cfg, seed=42):
    X, y = load_dataset(dataset_name)
    t0 = time.time()
    if algo == 'MOGSK':
        archive, _ = mogsk(X, y, backend=backend, seed=seed, **cfg)
    elif algo == 'NSGA-II':
        archive, _ = nsga2(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                            eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'MOPSO':
        archive, _ = mopso(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                            eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'MOEA/D':
        archive, _ = moead(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                            eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BPSO':
        archive, _ = bpso(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                           cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BHHO':
        archive, _ = bhho(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                           cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BGWO':
        archive, _ = bgwo(X, y, backend=backend, Np=cfg['Np'], T=cfg['T'],
                           cv_folds=cfg['cv_folds'], seed=seed)
    else:
        raise ValueError(f"Unknown algorithm: {algo}")
    elapsed = time.time() - t0
    return elapsed, len(archive)

cfg_main = dict(Np=Np, T=T, eta=1.0, tau=0.5, sigma_cauchy=0.5, beta_max=0.5,
                 cv_folds=CV_FOLDS, stagnation_limit=15)

all_runtime_records = []
csv_path = 'results/runtime_complexity.csv'

print("Measuring runtime with Checkpointing (this runs one instance of each algorithm per dataset/backend)...")
for ds in ALL_DATASETS:
    for backend in BACKENDS:
        ckpt = _load_runtime_ckpt(ds, backend)
        records = ckpt['records']
        done_algos = {r['algo'] for r in records}

        for algo in ALGOS:
            if algo in done_algos:
                rec = next(r for r in records if r['algo'] == algo)
                print(f"  {ds:<16} {backend:<4} {algo:<8} [SKIP] {rec['seconds']:7.2f}s  archive={rec['archive_size']}")
                all_runtime_records.append(rec)
                continue

            elapsed, n_arch = measure_runtime(ds, backend, algo, cfg_main)
            rec = {
                'dataset': ds,
                'backend': backend,
                'algo': algo,
                'seconds': round(elapsed, 2),
                'archive_size': n_arch
            }
            records.append(rec)
            all_runtime_records.append(rec)

            # حفظ فوري في ملف JSON الـ Checkpoint الخاص بالـ dataset والـ backend
            ckpt['records'] = records
            _save_runtime_ckpt(ds, backend, ckpt)

            # تحديث مباشر لملف الـ CSV الشامل بعد كل تجربة
            pd.DataFrame(all_runtime_records).to_csv(csv_path, index=False)
            print(f"  {ds:<16} {backend:<4} {algo:<8} {elapsed:7.2f}s  archive={n_arch}")

runtime_df = pd.DataFrame(all_runtime_records)
print(f"\n✓ Completed. Table saved to {csv_path}")
display(runtime_df)

Measuring runtime with Checkpointing (this runs one instance of each algorithm per dataset/backend)...
  SpamBase         SVM  MOGSK    [SKIP] 13192.08s  archive=17
  SpamBase         SVM  NSGA-II  [SKIP] 6923.84s  archive=40
  SpamBase         SVM  MOPSO    [SKIP] 11314.45s  archive=16
  SpamBase         SVM  MOEA/D   [SKIP] 20050.38s  archive=17
  SpamBase         SVM  BPSO     [SKIP] 10412.87s  archive=1
  SpamBase         SVM  BHHO     [SKIP] 11165.52s  archive=1
  SpamBase         SVM  BGWO     [SKIP] 3803.73s  archive=1
  SpamBase         KNN  MOGSK    [SKIP]  329.15s  archive=14
  SpamBase         KNN  NSGA-II  [SKIP]  408.21s  archive=40
  SpamBase         KNN  MOPSO    [SKIP]  419.79s  archive=131
  SpamBase         KNN  MOEA/D   [SKIP]  479.06s  archive=478
  SpamBase         KNN  BPSO     [SKIP]  146.33s  archive=1
  SpamBase         KNN  BHHO     [SKIP]  181.93s  archive=1
  SpamBase         KNN  BGWO     [SKIP]  151.10s  archive=1
  SpamBase         RF   MOGSK    [SKIP] 81

KeyboardInterrupt: 

## 5. Hyperparameter Search Space (addresses R2.5)

Explicit ranges used for each classifier backend's θ block.

In [46]:

hp_table = pd.DataFrame([
    {'Backend': 'SVM', 'theta_1': 'C', 'range_1': '[0.01, 100] (log-scale recommended)',
     'theta_2': 'gamma', 'range_2': '[0.0001, 10] (log-scale recommended)'},
    {'Backend': 'KNN', 'theta_1': 'k (n_neighbors)', 'range_1': '[1, 20] (int)',
     'theta_2': 'p (Minkowski)', 'range_2': '{1, 2} (int)'},
    {'Backend': 'RF',  'theta_1': 'n_estimators', 'range_1': '[10, 200] (int)',
     'theta_2': 'max_depth', 'range_2': '[2, 20] (int)'},
])
hp_table.to_csv('results/hyperparameter_ranges.csv', index=False)
hp_table


,Backend,theta_1,range_1,theta_2,range_2
0,SVM,C,"[0.01, 100] (log-scale recommended)",gamma,"[0.0001, 10] (log-scale recommended)"
1,KNN,k (n_neighbors),"[1, 20] (int)",p (Minkowski),"{1, 2} (int)"
2,RF,n_estimators,"[10, 200] (int)",max_depth,"[2, 20] (int)"


## 6. Validation Protocol (addresses R1.4.1, R2.4)

Explicit nested cross-validation, run once per dataset/backend/algorithm/run:

1. **Outer split**: `X, y` → `X_train, X_test` (stratified, `test_size=0.2`).
2. **Search / fitness (inner loop)**: `mogsk()` / baseline functions call
   `fitness()`, which runs `cv_folds`-fold CV *only on `X_train`*. The
   archive returned by the search therefore never touches `X_test`.
3. **Final scoring (outer loop, once)**: after search, the best archive
   member's `(theta, mask)` is refit on the *full* `X_train` and scored once
   on the untouched `X_test`. This is the number reported per run.

This separates the accuracy used to *drive* the search (inner CV, on
training data only) from the accuracy used to *report* results (outer test
fold, seen once), eliminating feature-selection leakage.

In [47]:

from sklearn.model_selection import train_test_split

def run_single_nested(algo, X, y, backend, cfg, seed):
    """One full nested run: outer split -> inner search -> outer test score."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)

    t0 = time.time()
    if algo == 'MOGSK':
        archive, hv_hist = mogsk(X_tr, y_tr, backend=backend, seed=seed, **cfg)
    elif algo == 'NSGA-II':
        archive, hv_hist = nsga2(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                  eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'MOPSO':
        archive, hv_hist = mopso(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                  eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'MOEA/D':
        archive, hv_hist = moead(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                  eta=cfg['eta'], tau=cfg['tau'], cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BPSO':
        archive, hv_hist = bpso(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                 cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BHHO':
        archive, hv_hist = bhho(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                 cv_folds=cfg['cv_folds'], seed=seed)
    elif algo == 'BGWO':
        archive, hv_hist = bgwo(X_tr, y_tr, backend=backend, Np=cfg['Np'], T=cfg['T'],
                                 cv_folds=cfg['cv_folds'], seed=seed)
    else:
        raise ValueError(algo)

    # Best archive member by inner-CV error -> refit on full X_tr, score on held-out X_te
    p = THETA_BOUNDS[backend.upper()].shape[0]
    best = min(archive, key=lambda a: a['f1'])
    theta, mask = best['z'][:p], best['m']
    X_tr_sel, X_te_sel = X_tr[:, mask == 1], X_te[:, mask == 1]
    clf = build_classifier(backend, theta)
    clf.fit(X_tr_sel, y_tr)
    test_acc = (clf.predict(X_te_sel) == y_te).mean()

    elapsed = time.time() - t0
    return {
        'algo': algo, 'backend': backend, 'seed': seed,
        'test_accuracy': round(test_acc * 100, 4),
        'n_feats': int(mask.sum()),
        'hv': round(hypervolume_2d(archive), 6),
        'hv_hist': hv_hist,
        'time_s': round(elapsed, 2),
    }

print("Nested-CV runner defined (outer test fold never touched during search).")


Nested-CV runner defined (outer test fold never touched during search).


## 7. Full Experiment — Tables 3–7 (all datasets × all backends, mean ± SD)

This is the main experiment loop: 10 datasets × 3 backends × 7 algorithms ×
`N_RUNS=31` runs, matching the paper's protocol (`Np=40`, `T=100`).

### Checkpointing / resume behaviour

Because the full run can take many hours, **every individual (dataset,
backend, algorithm, run) result is saved to disk immediately after it is
computed**, in `results/checkpoints/<dataset>_<backend>.json`. If the kernel
crashes, the machine is restarted, or you simply stop and re-run this cell
later:

- Already-completed `(algo, seed)` combinations are detected from the
  checkpoint file on disk and **skipped** — they are not recomputed.
- Only missing runs are executed.
- Mean/SD in the printed summary and in `master_results.json` are always
  computed from however many runs are actually present in the checkpoint
  (up to `N_RUNS`) — so partial progress is still usable, and once all
  `N_RUNS` are present the SD reflects the full 31-run sample as intended.

**To resume after an interruption, just re-run this cell (or re-run the
whole notebook) with the same `N_RUNS` — nothing needs to be deleted.**
To force a clean re-run from scratch for a given dataset/backend, delete its
corresponding file under `results/checkpoints/`.

In [48]:
import os
import json
import numpy as np


def _checkpoint_path(dataset_name, backend):
    safe = dataset_name.replace(' ', '_')
    return f'{CHECKPOINT_DIR}/{safe}_{backend}.json'


def _load_checkpoint(dataset_name, backend):
    path = _checkpoint_path(dataset_name, backend)

    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)

        # Backward compatibility:
        # لو الـ checkpoint القديم مفيهوش failed_runs
        if 'records' not in data:
            data['records'] = []

        if 'failed_runs' not in data:
            data['failed_runs'] = []

        return data

    return {
        'records': [],
        'failed_runs': []
    }


def _save_checkpoint(dataset_name, backend, data):
    """
    Atomic write:
    بنكتب الأول في ملف مؤقت، وبعدها نستبدل الملف الأصلي.
    بالتالي لو البرنامج وقف فجأة أثناء الحفظ،
    الـ checkpoint القديم يفضل سليم.
    """
    path = _checkpoint_path(dataset_name, backend)
    tmp = path + '.tmp'

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(tmp, 'w') as f:
        json.dump(
            data,
            f,
            indent=2,
            default=_json_convert
        )

    os.replace(tmp, path)


def run_experiment_resumable(
    dataset_name,
    backend,
    algos,
    cfg,
    n_runs
):
    X, y = load_dataset(dataset_name)

    d_total = X.shape[1]

    print(
        f"\n=== {dataset_name} | {backend} | "
        f"d={d_total} | target n_runs={n_runs} ==="
    )

    # =========================================================
    # Load existing checkpoint
    # =========================================================

    ckpt = _load_checkpoint(dataset_name, backend)

    records = ckpt.get('records', [])
    failed_runs = ckpt.get('failed_runs', [])

    # Successful runs already completed
    done_keys = {
        (r['algo'], r['seed'])
        for r in records
    }

    # Failed runs already attempted
    failed_keys = {
        (r['algo'], r['seed'])
        for r in failed_runs
    }

    print(
        f"  Loaded checkpoint: "
        f"{len(records)} successful runs, "
        f"{len(failed_runs)} failed/skipped runs"
    )

    # =========================================================
    # Experiments
    # =========================================================

    for algo in algos:

        n_success = sum(
            1
            for r in records
            if r['algo'] == algo
        )

        n_failed = sum(
            1
            for r in failed_runs
            if r['algo'] == algo
        )

        # لو عندنا بالفعل العدد المطلوب من النتائج الناجحة
        if n_success >= n_runs:

            print(
                f"  {algo:<8} [SKIP] "
                f"{n_success}/{n_runs} successful runs "
                f"already checkpointed"
            )

            continue

        # =====================================================
        # Run seeds
        # =====================================================

        for run in range(n_runs):

            seed = 42 + run

            key = (algo, seed)

            # -----------------------------------------
            # Successful run موجود بالفعل
            # -----------------------------------------

            if key in done_keys:

                print(
                    f"  {algo:<8} "
                    f"seed={seed} [ALREADY DONE]"
                )

                continue

            # -----------------------------------------
            # Failed run موجود بالفعل
            # -----------------------------------------

            if key in failed_keys:

                print(
                    f"  {algo:<8} "
                    f"seed={seed} [PREVIOUSLY FAILED - SKIP]"
                )

                continue

            # -----------------------------------------
            # New run
            # -----------------------------------------

            try:

                print(
                    f"  {algo:<8} "
                    f"seed={seed} [RUNNING]"
                )

                rec = run_single_nested(
                    algo,
                    X,
                    y,
                    backend,
                    cfg,
                    seed
                )

                # تأكد إن البيانات المهمة موجودة
                rec['algo'] = algo
                rec['seed'] = seed

                # Save successful result
                records.append(rec)

                done_keys.add(key)

                ckpt['records'] = records
                ckpt['failed_runs'] = failed_runs

                # حفظ فورًا بعد كل run
                _save_checkpoint(
                    dataset_name,
                    backend,
                    ckpt
                )

                print(
                    f"  {algo:<8} "
                    f"seed={seed} [SAVED]"
                )

            # =================================================
            # Skip ValueError
            # زي مشكلة KNN:
            # n_neighbors > n_samples_fit
            # =================================================

            except ValueError as e:

                print(
                    f"\n  {algo:<8} "
                    f"seed={seed} [FAILED -> SKIP]"
                )

                print(
                    f"      Error: {e}"
                )

                failed_record = {
                    'algo': algo,
                    'seed': seed,
                    'error_type': 'ValueError',
                    'error': str(e)
                }

                failed_runs.append(
                    failed_record
                )

                failed_keys.add(key)

                ckpt['records'] = records
                ckpt['failed_runs'] = failed_runs

                # مهم:
                # نسجل الـ failed run عشان بعد restart
                # مايحاولش يعمله من جديد
                _save_checkpoint(
                    dataset_name,
                    backend,
                    ckpt
                )

                continue

        # =====================================================
        # Algo status
        # =====================================================

        n_success = sum(
            1
            for r in records
            if r['algo'] == algo
        )

        n_failed = sum(
            1
            for r in failed_runs
            if r['algo'] == algo
        )

        print(
            f"\n  {algo:<8} [STATUS] "
            f"successful={n_success}/{n_runs}, "
            f"failed/skipped={n_failed}"
        )

    # =========================================================
    # Build summary
    # فقط من الـ successful runs
    # =========================================================

    summary = {}

    for algo in algos:

        algo_records = [
            r
            for r in records
            if r['algo'] == algo
        ]

        accs = [
            r['test_accuracy']
            for r in algo_records
        ]

        hvs = [
            r['hv']
            for r in algo_records
        ]

        nf = [
            r['n_feats']
            for r in algo_records
        ]

        hv_hists = [
            r['hv_hist']
            for r in algo_records
        ]

        same_len = (
            len(set(len(h) for h in hv_hists)) == 1
            if hv_hists
            else False
        )

        n_failed = sum(
            1
            for r in failed_runs
            if r['algo'] == algo
        )

        summary[algo] = {

            'acc_mean': (
                round(float(np.mean(accs)), 2)
                if accs
                else None
            ),

            'acc_std': (
                round(float(np.std(accs)), 2)
                if accs
                else None
            ),

            'hv_mean': (
                round(float(np.mean(hvs)), 4)
                if hvs
                else None
            ),

            'hv_std': (
                round(float(np.std(hvs)), 4)
                if hvs
                else None
            ),

            'nfeats_mean': (
                round(float(np.mean(nf)), 1)
                if nf
                else None
            ),

            'n_runs_completed': len(algo_records),

            'n_runs_failed': n_failed,

            'hv_hist_mean': (
                list(
                    np.mean(
                        hv_hists,
                        axis=0
                    )
                )
                if same_len
                else []
            ),
        }

        s = summary[algo]

        # عشان لو كل الـ runs فشلت، مايحصلش formatting error
        if s['acc_mean'] is not None:

            print(
                f"  {algo:<8} "
                f"Acc={s['acc_mean']:.2f}"
                f"±{s['acc_std']:.2f}%  "
                f"HV={s['hv_mean']:.4f}"
                f"±{s['hv_std']:.4f}  "
                f"Feats={s['nfeats_mean']:.1f}/{d_total}  "
                f"(success={s['n_runs_completed']}/{n_runs}, "
                f"failed={s['n_runs_failed']})"
            )

        else:

            print(
                f"  {algo:<8} "
                f"No successful runs "
                f"(success=0/{n_runs}, "
                f"failed={s['n_runs_failed']})"
            )

    return (
        records,
        summary,
        d_total
    )


# =============================================================
# MAIN EXPERIMENT LOOP
# =============================================================

master_results = {}

for ds in ALL_DATASETS:

    master_results[ds] = {}

    for backend in BACKENDS:

        records, summary, d_total = run_experiment_resumable(
            ds,
            backend,
            ALGOS,
            cfg_main,
            N_RUNS
        )

        master_results[ds][backend] = {
            'summary': summary,
            'd_total': d_total,
            'records': records
        }


# =============================================================
# Save master results
# =============================================================

os.makedirs(
    'results',
    exist_ok=True
)

with open(
    'results/master_results.json',
    'w'
) as f:

    json.dump(
        master_results,
        f,
        indent=2,
        default=_json_convert
    )


print(
    "\n✓ Saved results/master_results.json"
)

print(
    "✓ Existing successful checkpoints were preserved."
)

print(
    "✓ Failed ValueError runs were recorded and will be skipped "
    "on future restarts."
)

print(
    "✓ Per-dataset/backend checkpoints live under "
    "results/checkpoints/"
)


=== SpamBase | SVM | d=57 | target n_runs=11 ===
  Loaded checkpoint: 77 successful runs, 0 failed/skipped runs
  MOGSK    [SKIP] 11/11 successful runs already checkpointed
  NSGA-II  [SKIP] 11/11 successful runs already checkpointed
  MOPSO    [SKIP] 11/11 successful runs already checkpointed
  MOEA/D   [SKIP] 11/11 successful runs already checkpointed
  BPSO     [SKIP] 11/11 successful runs already checkpointed
  BHHO     [SKIP] 11/11 successful runs already checkpointed
  BGWO     [SKIP] 11/11 successful runs already checkpointed
  MOGSK    Acc=92.31±1.00%  HV=0.7197±0.0342  Feats=35.6/57  (success=11/11, failed=0)
  NSGA-II  Acc=91.39±1.35%  HV=0.8764±0.0208  Feats=13.5/57  (success=11/11, failed=0)
  MOPSO    Acc=92.54±0.79%  HV=0.7811±0.0333  Feats=28.9/57  (success=11/11, failed=0)
  MOEA/D   Acc=91.25±1.08%  HV=0.6803±0.0393  Feats=24.8/57  (success=11/11, failed=0)
  BPSO     Acc=92.33±0.43%  HV=0.4406±0.0339  Feats=30.0/57  (success=11/11, failed=0)
  BHHO     Acc=91.06±0.72

### 7a. Table 3 — Accuracy & Feature Reduction (SVM backend), with corrected mean arithmetic

In [49]:

rows = []
for ds in ALL_DATASETS:
    s = master_results[ds]['SVM']['summary']
    d_total = master_results[ds]['SVM']['d_total']
    all_feat_acc = s.get('BPSO', {}).get('acc_mean', np.nan)  # reference "all/near-all features" baseline proxy
    mogsk_acc = s['MOGSK']['acc_mean']
    sel_feats = s['MOGSK']['nfeats_mean']
    reduction = (1 - sel_feats / d_total) * 100
    rows.append({
        'Dataset': ds, 'Acc_MOGSK_mean': mogsk_acc, 'Acc_MOGSK_std': s['MOGSK']['acc_std'],
        'Sel_Feats_mean': sel_feats, 'Orig_Feats': d_total, 'Reduction_%': round(reduction, 1),
    })
table3 = pd.DataFrame(rows)
table3.loc['MEAN'] = table3.mean(numeric_only=True)
table3.to_csv('results/table3_accuracy_SVM.csv')
print(f"Mean Reduction%% (recomputed, not hard-coded): {table3.loc['MEAN', 'Reduction_%']:.2f}%%")
table3


Mean Reduction%% (recomputed, not hard-coded): 50.97%%


,Dataset,Acc_MOGSK_mean,Acc_MOGSK_std,Sel_Feats_mean,Orig_Feats,Reduction_%
0,SpamBase,92.310,1.000,35.60,57.0,37.50
1,Student Dropout,33.850,1.340,13.20,35.0,62.30
2,Optical Digits,94.970,6.600,34.40,64.0,46.20
3,Lung Cancer,37.880,10.280,23.10,56.0,58.70
4,Heart Disease,82.420,5.090,6.50,13.0,50.00
5,Breast Cancer,94.740,1.750,16.20,30.0,46.00
6,Wine Quality,73.920,2.200,6.20,11.0,43.60
7,Ionosphere,91.290,2.310,15.30,34.0,55.00
8,Sonar,77.060,10.170,29.90,60.0,50.20
9,Arrhythmia,56.240,7.580,109.10,274.0,60.20


### 7b. Table 7 — Hypervolume, grouped by classifier backend (addresses R1.5.4)

In [50]:

for backend in BACKENDS:
    print(f"\n--- HV Table — {backend} backend ---")
    rows = []
    for ds in ALL_DATASETS:
        s = master_results[ds][backend]['summary']
        row = {'Dataset': ds}
        for algo in ALGOS:
            row[algo] = f"{s[algo]['hv_mean']:.4f} ± {s[algo]['hv_std']:.4f}"
        rows.append(row)
    df_backend = pd.DataFrame(rows)
    df_backend.to_csv(f'results/table7_hypervolume_{backend}.csv', index=False)
    display(df_backend)



--- HV Table — SVM backend ---


,Dataset,MOGSK,NSGA-II,MOPSO,MOEA/D,BPSO,BHHO,BGWO
0,SpamBase,0.7197 ± 0.0342,0.8764 ± 0.0208,0.7811 ± 0.0333,0.6803 ± 0.0393,0.4406 ± 0.0339,0.3912 ± 0.0858,0.5848 ± 0.0185
1,Student Dropout,0.2618 ± 0.0198,0.2836 ± 0.0136,0.2740 ± 0.0206,0.2443 ± 0.0224,0.1957 ± 0.0356,0.2631 ± 0.0933,0.2147 ± 0.0293
2,Optical Digits,0.5977 ± 0.0596,0.6280 ± 0.1229,0.6495 ± 0.0327,0.5412 ± 0.1176,0.4332 ± 0.0815,0.4058 ± 0.0864,0.5067 ± 0.0354
3,Lung Cancer,0.3702 ± 0.0784,0.3799 ± 0.0679,0.3235 ± 0.0360,0.3070 ± 0.0455,0.2920 ± 0.0574,0.5069 ± 0.0656,0.4167 ± 0.0988
4,Heart Disease,0.6452 ± 0.0705,0.7201 ± 0.0434,0.6993 ± 0.0496,0.5791 ± 0.0600,0.2889 ± 0.1076,0.2839 ± 0.1279,0.3820 ± 0.0695
5,Breast Cancer,0.7047 ± 0.0419,0.8038 ± 0.0459,0.7548 ± 0.0559,0.6918 ± 0.0517,0.5131 ± 0.0798,0.5278 ± 0.1155,0.5600 ± 0.0629
6,Wine Quality,0.6018 ± 0.0500,0.6416 ± 0.0380,0.6332 ± 0.0319,0.5589 ± 0.0567,0.1762 ± 0.0681,0.1178 ± 0.1189,0.2486 ± 0.1154
7,Ionosphere,0.6855 ± 0.0419,0.7593 ± 0.0422,0.6844 ± 0.0334,0.6153 ± 0.0497,0.4230 ± 0.0698,0.4069 ± 0.0760,0.5042 ± 0.0389
8,Sonar,0.5413 ± 0.0560,0.5605 ± 0.0646,0.5619 ± 0.0823,0.4994 ± 0.0759,0.4778 ± 0.0389,0.4768 ± 0.0781,0.5411 ± 0.0504
9,Arrhythmia,0.3428 ± 0.0278,0.4008 ± 0.0604,0.3842 ± 0.0637,0.3608 ± 0.0566,0.3860 ± 0.0124,0.6298 ± 0.0253,0.4769 ± 0.0292



--- HV Table — KNN backend ---


,Dataset,MOGSK,NSGA-II,MOPSO,MOEA/D,BPSO,BHHO,BGWO
0,SpamBase,0.7374 ± 0.0274,0.8438 ± 0.0247,0.7985 ± 0.0382,0.7404 ± 0.0489,0.4412 ± 0.0543,0.3560 ± 0.1234,0.6216 ± 0.0396
1,Student Dropout,0.2619 ± 0.0107,0.2774 ± 0.0181,0.2650 ± 0.0161,0.2446 ± 0.0167,0.1789 ± 0.0305,0.1966 ± 0.0853,0.2028 ± 0.0252
2,Optical Digits,0.6440 ± 0.0342,0.6762 ± 0.0273,0.6674 ± 0.0378,0.6228 ± 0.0367,0.3838 ± 0.0448,0.0526 ± 0.0643,0.4587 ± 0.0433
3,Lung Cancer,0.4161 ± 0.0254,0.5421 ± 0.0669,0.4784 ± 0.0383,0.4387 ± 0.0337,0.3870 ± 0.0552,0.4050 ± 0.1267,0.4589 ± 0.0545
4,Heart Disease,0.6669 ± 0.0389,0.6927 ± 0.0607,0.6783 ± 0.0731,0.6341 ± 0.0600,0.3177 ± 0.1136,0.2922 ± 0.1349,0.3019 ± 0.0896
5,Breast Cancer,0.7294 ± 0.0330,0.8092 ± 0.0639,0.7575 ± 0.0580,0.7346 ± 0.0330,0.4755 ± 0.0728,0.4753 ± 0.0986,0.5245 ± 0.0749
6,Wine Quality,0.6247 ± 0.0384,0.6343 ± 0.0434,0.6243 ± 0.0474,0.5675 ± 0.0464,0.3033 ± 0.1404,0.3434 ± 0.1160,0.3795 ± 0.0917
7,Ionosphere,0.6803 ± 0.0395,0.7437 ± 0.0284,0.6894 ± 0.0434,0.6338 ± 0.0452,0.5355 ± 0.0671,0.6133 ± 0.1661,0.5980 ± 0.0765
8,Sonar,0.5876 ± 0.0230,0.6399 ± 0.0368,0.5979 ± 0.0451,0.5461 ± 0.0363,0.4498 ± 0.0481,0.4467 ± 0.1006,0.5100 ± 0.0734
9,Arrhythmia,0.4037 ± 0.0162,0.4372 ± 0.0167,0.4188 ± 0.0207,0.4014 ± 0.0122,0.3596 ± 0.0238,0.4649 ± 0.1022,0.4403 ± 0.0237



--- HV Table — RF backend ---


,Dataset,MOGSK,NSGA-II,MOPSO,MOEA/D,BPSO,BHHO,BGWO
0,SpamBase,0.7189 ± 0.0596,0.6733 ± 0.0363,0.6572 ± 0.0421,0.6189 ± 0.0377,0.4015 ± 0.0562,0.1449 ± 0.1035,0.4653 ± 0.0453
1,Student Dropout,0.2601 ± 0.0101,0.2801 ± 0.0155,0.2729 ± 0.0159,0.2490 ± 0.0174,0.1909 ± 0.0289,0.1916 ± 0.0434,0.2394 ± 0.0293
2,Optical Digits,0.6373 ± 0.0374,0.6837 ± 0.0346,0.6641 ± 0.0377,0.6157 ± 0.0331,0.3591 ± 0.0438,0.0612 ± 0.0529,0.4307 ± 0.0385
3,Lung Cancer,0.4964 ± 0.0500,0.5755 ± 0.0557,0.5224 ± 0.0365,0.4562 ± 0.0391,0.4056 ± 0.0623,0.4991 ± 0.1588,0.4621 ± 0.0635
4,Heart Disease,0.6521 ± 0.0683,0.7134 ± 0.0510,0.6939 ± 0.0442,0.6203 ± 0.0769,0.3483 ± 0.0748,0.3439 ± 0.1039,0.3449 ± 0.1012
5,Breast Cancer,0.7221 ± 0.0451,0.8091 ± 0.0425,0.7649 ± 0.0675,0.7206 ± 0.0481,0.4906 ± 0.0755,0.5020 ± 0.1081,0.5578 ± 0.0595
6,Wine Quality,0.6687 ± 0.0300,0.6519 ± 0.0410,0.6605 ± 0.0634,0.6322 ± 0.0502,0.2474 ± 0.0960,0.1693 ± 0.1143,0.3029 ± 0.1065
7,Ionosphere,0.6907 ± 0.0477,0.7369 ± 0.0464,0.6941 ± 0.0425,0.6596 ± 0.0566,0.4635 ± 0.0890,0.3825 ± 0.1531,0.4673 ± 0.1104
8,Sonar,0.5607 ± 0.0371,0.6248 ± 0.0317,0.5829 ± 0.0422,0.5400 ± 0.0217,0.4148 ± 0.0633,0.3474 ± 0.1381,0.4754 ± 0.0489
9,Arrhythmia,0.4723 ± 0.0213,0.5040 ± 0.0216,0.4869 ± 0.0208,0.4735 ± 0.0166,0.3997 ± 0.0217,0.3425 ± 0.1258,0.4813 ± 0.0159


### 7c. Table 10 — Wilcoxon Signed-Rank Tests on **all 8 datasets** (addresses R1.5.6)

In [51]:

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

wilcoxon_rows = []
for backend in BACKENDS:
    for ds in ALL_DATASETS:
        records = master_results[ds][backend]['records']
        by_algo = {}
        for r in records:
            by_algo.setdefault(r['algo'], []).append(r['test_accuracy'])
        mogsk_vals = by_algo.get('MOGSK', [])
        for algo in ['NSGA-II', 'MOPSO', 'MOEA/D', 'BPSO', 'BHHO', 'BGWO']:
            vals = by_algo.get(algo, [])
            if len(mogsk_vals) >= 2 and len(vals) == len(mogsk_vals) and any(m != v for m, v in zip(mogsk_vals, vals)):
                try:
                    stat, p = wilcoxon(mogsk_vals, vals, alternative='greater')
                except Exception:
                    p = np.nan
            else:
                p = np.nan
            wilcoxon_rows.append({'Backend': backend, 'Dataset': ds, 'Comparator': algo, 'p_raw': p})

wilcoxon_df = pd.DataFrame(wilcoxon_rows).dropna(subset=['p_raw'])
if len(wilcoxon_df):
    reject, p_corrected, _, _ = multipletests(wilcoxon_df['p_raw'], alpha=0.05, method='bonferroni')
    wilcoxon_df['p_bonferroni'] = p_corrected
    wilcoxon_df['significant'] = reject
wilcoxon_df.to_csv('results/table10_wilcoxon_all8.csv', index=False)
wilcoxon_df


,Backend,Dataset,Comparator,p_raw,p_bonferroni,significant
0,SVM,SpamBase,NSGA-II,0.011719,1.000000,False
1,SVM,SpamBase,MOPSO,0.652344,1.000000,False
2,SVM,SpamBase,MOEA/D,0.018555,1.000000,False
3,SVM,SpamBase,BPSO,0.374512,1.000000,False
4,SVM,SpamBase,BHHO,0.001465,0.256348,False
...,...,...,...,...,...,...
175,RF,Arrhythmia,MOPSO,0.765625,1.000000,False
176,RF,Arrhythmia,MOEA/D,0.254883,1.000000,False
177,RF,Arrhythmia,BPSO,0.147461,1.000000,False
178,RF,Arrhythmia,BHHO,0.304688,1.000000,False


## 8. Table 8 — Ablation Study, all 10 datasets (addresses R1.5.5, R2.9)

Baseline (default hyperparams, all features) vs. FS-only vs. PT-only vs.
Joint MOGSK, with the corrected superadditivity-gap formula:

`Gap = (Joint_gain) − (FS_gain + PT_gain)`, where each `_gain` is relative
to the Baseline mean.

### Checkpointing

Same resumable pattern as Section 7: per-run results for each
(dataset, backend) combination are saved to
`results/checkpoints/ablation_<dataset>_<backend>.json` after every run, and
already-completed runs are skipped on re-execution.

In [52]:

def baseline_acc(X, y, backend, cv_folds=5):
    theta = {'SVM': [1.0, 0.1], 'KNN': [5.0, 2.0], 'RF': [100.0, 5.0]}[backend.upper()]
    clf = build_classifier(backend, np.array(theta))
    return cross_val_score(clf, X, y, cv=cv_folds, scoring='accuracy').mean() * 100

def mogsk_fs_only(X, y, backend, Np, T, cv_folds, seed):
    theta_fixed = np.array({'SVM': [1.0, 0.1], 'KNN': [5.0, 2.0], 'RF': [100.0, 5.0]}[backend.upper()])
    rng = np.random.default_rng(seed)
    d = X.shape[1]; p = THETA_BOUNDS[backend.upper()].shape[0]
    V = rng.uniform(-3, 3, (Np, d))
    def _eval(v):
        z = np.concatenate([theta_fixed, v])
        f1, f2, m = fitness(z, X, y, backend, p, cv_folds, 1.0, 0.5)
        return {'z': z.copy(), 'f1': f1, 'f2': f2, 'm': m}
    pop = [_eval(V[i]) for i in range(Np)]
    archive = []
    for s in pop: archive = update_archive(archive, s)
    for t in range(T):
        for i in range(Np):
            z_mentor = select_mentor(archive)[p:]
            alpha = rng.uniform(); delta = rng.standard_cauchy(d) * 0.5
            V[i] = np.clip(V[i] + alpha*(z_mentor - V[i]) + delta, -10, 10)
            archive = update_archive(archive, _eval(V[i]))
    return (1 - min(a['f1'] for a in archive)) * 100

def mogsk_pt_only(X, y, backend, Np, T, cv_folds, seed):
    rng = np.random.default_rng(seed)
    p = THETA_BOUNDS[backend.upper()].shape[0]
    lb, ub = THETA_BOUNDS[backend.upper()][:, 0], THETA_BOUNDS[backend.upper()][:, 1]
    def _eval(theta):
        clf = build_classifier(backend, theta)
        return cross_val_score(clf, X, y, cv=cv_folds, scoring='accuracy').mean() * 100
    Theta = rng.uniform(lb, ub, (Np, p))
    best = max(_eval(Theta[i]) for i in range(Np))
    for t in range(T):
        for i in range(Np):
            j, k = rng.choice(Np, 2, replace=False)
            child = np.clip(Theta[i] + 0.8*(Theta[j]-Theta[k]) + rng.normal(0, 0.1, p), lb, ub)
            acc = _eval(child)
            if acc > best: best = acc; Theta[i] = child
    return best

def _ablation_checkpoint_path(dataset_name, backend):
    safe = dataset_name.replace(' ', '_')
    return f'{CHECKPOINT_DIR}/ablation_{safe}_{backend}.json'

def _load_ablation_checkpoint(dataset_name, backend):
    path = _ablation_checkpoint_path(dataset_name, backend)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {'runs': []}  # each entry: {'seed', 'base', 'fs', 'pt', 'joint'}

def _save_ablation_checkpoint(dataset_name, backend, data):
    path = _ablation_checkpoint_path(dataset_name, backend)
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(data, f, indent=2, default=_json_convert)
    os.replace(tmp, path)

ablation_rows = []
for ds in ALL_DATASETS:
    X, y = load_dataset(ds)
    for backend in BACKENDS:
        ckpt = _load_ablation_checkpoint(ds, backend)
        done_seeds = {r['seed'] for r in ckpt['runs']}

        n_already = len(ckpt['runs'])
        if n_already >= N_RUNS:
            print(f"{ds:<16} {backend:<4} [SKIP] {n_already}/{N_RUNS} ablation runs already checkpointed")
        else:
            for r in range(N_RUNS):
                seed = 42 + r
                if seed in done_seeds:
                    continue
                base = baseline_acc(X, y, backend, CV_FOLDS)
                fs   = mogsk_fs_only(X, y, backend, Np, T, CV_FOLDS, seed)
                pt   = mogsk_pt_only(X, y, backend, Np, T, CV_FOLDS, seed)
                archive, _ = mogsk(X, y, backend=backend, Np=Np, T=T, cv_folds=CV_FOLDS, seed=seed)
                joint = (1 - min(a['f1'] for a in archive)) * 100
                ckpt['runs'].append({'seed': seed, 'base': base, 'fs': fs, 'pt': pt, 'joint': joint})
                done_seeds.add(seed)
                # Save after every single run
                _save_ablation_checkpoint(ds, backend, ckpt)
            print(f"{ds:<16} {backend:<4} [DONE] {len(ckpt['runs'])}/{N_RUNS} ablation runs checkpointed")

        bases  = [r['base']  for r in ckpt['runs']]
        fss    = [r['fs']    for r in ckpt['runs']]
        pts    = [r['pt']    for r in ckpt['runs']]
        joints = [r['joint'] for r in ckpt['runs']]
        base_m, fs_m, pt_m, jt_m = np.mean(bases), np.mean(fss), np.mean(pts), np.mean(joints)
        naive_sum = (fs_m - base_m) + (pt_m - base_m)
        gap = (jt_m - base_m) - naive_sum
        ablation_rows.append({
            'Dataset': ds, 'Backend': backend, 'N_runs': len(ckpt['runs']),
            'Baseline_mean': round(base_m, 2), 'Baseline_std': round(np.std(bases), 2),
            'FS_only_mean': round(fs_m, 2), 'FS_only_std': round(np.std(fss), 2),
            'PT_only_mean': round(pt_m, 2), 'PT_only_std': round(np.std(pts), 2),
            'Joint_mean': round(jt_m, 2), 'Joint_std': round(np.std(joints), 2),
            'Superadditivity_Gap_pp': round(gap, 2),
        })
        print(f"  -> {ds:<16} {backend:<4} Gap={gap:+.2f}pp  (n={len(ckpt['runs'])})")

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv('results/table8_ablation_all8.csv', index=False)
ablation_df


SpamBase         SVM  [SKIP] 11/11 ablation runs already checkpointed
  -> SpamBase         SVM  Gap=-4.00pp  (n=11)
SpamBase         KNN  [SKIP] 11/11 ablation runs already checkpointed
  -> SpamBase         KNN  Gap=-0.37pp  (n=11)
SpamBase         RF   [SKIP] 11/11 ablation runs already checkpointed
  -> SpamBase         RF   Gap=+0.73pp  (n=11)
Student Dropout  SVM  [SKIP] 11/11 ablation runs already checkpointed
  -> Student Dropout  SVM  Gap=-1.10pp  (n=11)
Student Dropout  KNN  [SKIP] 11/11 ablation runs already checkpointed
  -> Student Dropout  KNN  Gap=-0.27pp  (n=11)
Student Dropout  RF   [SKIP] 11/11 ablation runs already checkpointed
  -> Student Dropout  RF   Gap=-1.40pp  (n=11)
Optical Digits   SVM  [SKIP] 11/11 ablation runs already checkpointed
  -> Optical Digits   SVM  Gap=-5.72pp  (n=11)
Optical Digits   KNN  [SKIP] 11/11 ablation runs already checkpointed
  -> Optical Digits   KNN  Gap=-0.33pp  (n=11)
Optical Digits   RF   [SKIP] 11/11 ablation runs already checkpo

,Dataset,Backend,N_runs,Baseline_mean,Baseline_std,FS_only_mean,FS_only_std,PT_only_mean,PT_only_std,Joint_mean,Joint_std,Superadditivity_Gap_pp
0,SpamBase,SVM,11,89.02,0.0,92.25,0.34,91.81,0.00,91.03,2.07,-4.00
1,SpamBase,KNN,11,88.37,0.0,90.96,0.45,89.16,0.05,91.38,0.35,-0.37
2,SpamBase,RF,11,91.35,0.0,92.45,0.40,93.05,0.02,94.88,0.14,0.73
3,Student Dropout,SVM,11,34.63,0.0,36.49,0.60,34.37,0.39,35.13,0.83,-1.10
4,Student Dropout,KNN,11,32.98,0.0,35.39,0.34,33.77,0.00,35.91,0.32,-0.27
5,Student Dropout,RF,11,34.02,0.0,36.44,0.34,35.17,0.18,36.18,0.31,-1.40
6,Optical Digits,SVM,11,94.84,0.0,98.13,0.22,97.63,0.11,95.20,7.58,-5.72
7,Optical Digits,KNN,11,97.26,0.0,97.46,0.34,97.70,0.00,97.57,0.40,-0.33
8,Optical Digits,RF,11,93.27,0.0,92.65,0.48,97.97,0.02,97.84,0.21,0.50
9,Lung Cancer,SVM,11,37.33,0.0,65.94,5.70,47.70,0.96,46.85,7.14,-29.45


## 9. Table 9 — Sensitivity Analysis (SpamBase, SVM, as in the paper)

In [53]:

def run_sensitivity(dataset_name, backend, runs):
    X, y = load_dataset(dataset_name)
    base_cfg = dict(Np=Np, T=T, eta=1.0, tau=0.5, sigma_cauchy=0.5,
                     beta_max=0.5, cv_folds=CV_FOLDS, stagnation_limit=5)
    params = {
        'tau':          ([0.3, 0.4, 0.5, 0.6, 0.7], 'Binarization threshold'),
        'eta':          ([0.1, 0.3, 0.5, 0.7, 1.0], 'Sigmoid steepness'),
        'Np':           ([20, 40, 60, 80, 100],     'Population size'),
        'T':            ([30, 50, 70, 100, 150],    'Max iterations'),
        'sigma_cauchy': ([0.1, 0.5, 1.0, 1.5, 2.0], 'Cauchy scale'),
        'beta_max':     ([0.1, 0.3, 0.5, 0.7],      'Senior beta'),
    }
    sens_rows = []
    for param, (values, label) in params.items():
        for val in values:
            cfg = {**base_cfg, param: val}
            accs = []
            for r in range(runs):
                archive, _ = mogsk(X, y, backend=backend, seed=42+r, **cfg)
                accs.append((1 - min(a['f1'] for a in archive)) * 100)
            sens_rows.append({'Param': param, 'Label': label, 'Value': val,
                               'Acc_mean': round(np.mean(accs), 2), 'Acc_std': round(np.std(accs), 2)})
    return pd.DataFrame(sens_rows)

sensitivity_df = run_sensitivity('SpamBase', 'SVM', runs=max(2, N_RUNS // 10))
sensitivity_df.to_csv('results/table9_sensitivity.csv', index=False)
sensitivity_df


,Param,Label,Value,Acc_mean,Acc_std
0,tau,Binarization threshold,0.3,90.38,0.49
1,tau,Binarization threshold,0.4,91.31,0.20
2,tau,Binarization threshold,0.5,90.64,0.77
3,tau,Binarization threshold,0.6,90.81,0.04
4,tau,Binarization threshold,0.7,89.77,1.01
5,eta,Sigmoid steepness,0.1,90.64,0.77
6,eta,Sigmoid steepness,0.3,90.64,0.77
7,eta,Sigmoid steepness,0.5,90.64,0.77
8,eta,Sigmoid steepness,0.7,90.64,0.77
9,eta,Sigmoid steepness,1.0,90.64,0.77


## 10. Summary / Sanity Checks

Quick automated checks so any future arithmetic slip (like the Table 3
Reduction% / Table 8 Gap issues found during review) is caught immediately
rather than surviving into the manuscript.

In [54]:

# Sanity check 1: Table 3 mean Reduction% is computed, not typed by hand.
computed_mean_reduction = table3.loc['MEAN', 'Reduction_%']
print(f"[CHECK] Table 3 mean Reduction%%: {computed_mean_reduction:.2f}%% (recomputed from the table itself)")

# Sanity check 2: ablation gap sign is derived, not asserted.
for _, row in ablation_df.iterrows():
    sign = '+' if row['Superadditivity_Gap_pp'] >= 0 else '-'
    print(f"[CHECK] {row['Dataset']:<16} {row['Backend']:<4} Gap sign: {sign} "
          f"({row['Superadditivity_Gap_pp']:+.2f} pp)")

print("\n✓ All tables saved under results/. Review each CSV before pasting into the manuscript.")


[CHECK] Table 3 mean Reduction%%: 50.97%% (recomputed from the table itself)
[CHECK] SpamBase         SVM  Gap sign: - (-4.00 pp)
[CHECK] SpamBase         KNN  Gap sign: - (-0.37 pp)
[CHECK] SpamBase         RF   Gap sign: + (+0.73 pp)
[CHECK] Student Dropout  SVM  Gap sign: - (-1.10 pp)
[CHECK] Student Dropout  KNN  Gap sign: - (-0.27 pp)
[CHECK] Student Dropout  RF   Gap sign: - (-1.40 pp)
[CHECK] Optical Digits   SVM  Gap sign: - (-5.72 pp)
[CHECK] Optical Digits   KNN  Gap sign: - (-0.33 pp)
[CHECK] Optical Digits   RF   Gap sign: + (+0.50 pp)
[CHECK] Lung Cancer      SVM  Gap sign: - (-29.45 pp)
[CHECK] Lung Cancer      KNN  Gap sign: - (-9.52 pp)
[CHECK] Lung Cancer      RF   Gap sign: - (-13.15 pp)
[CHECK] Heart Disease    SVM  Gap sign: - (-4.01 pp)
[CHECK] Heart Disease    KNN  Gap sign: - (-0.18 pp)
[CHECK] Heart Disease    RF   Gap sign: - (-3.37 pp)
[CHECK] Breast Cancer    SVM  Gap sign: - (-3.12 pp)
[CHECK] Breast Cancer    KNN  Gap sign: - (-0.51 pp)
[CHECK] Breast Cance

## 11. Figures

All figures are generated from `master_results` (the actual data computed in
Section 7) and saved to `plots/` as PNG files. Nothing here is copied from
the manuscript — every figure is redrawn from the numbers this notebook
computed.

- **Fig. 2 equivalent** — Pareto front comparison (MOGSK vs. NSGA-II vs.
  MOPSO vs. BPSO) for a representative dataset/backend.
- **Fig. 3 equivalent** — HV bar chart across the primary datasets (SVM
  backend), all 7 algorithms.
- **Fig. 4 equivalent** — Convergence curves (best archive accuracy vs.
  generation) for MOGSK (per backend) vs. comparators.
- **Fig. 5 equivalent** — MOGSK accuracy across all 10 datasets × 3
  backends.
- **Fig. 6 equivalent** — Feature count / reduction % per dataset (SVM
  backend), with the *computed* mean line (not the hard-coded 77.3%).

In [55]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

COLORS = {'MOGSK':'#1565C0','NSGA-II':'#E65100','MOPSO':'#2E7D32',
          'MOEA/D':'#6A1B9A','BPSO':'#B71C1C','BHHO':'#00838F','BGWO':'#F9A825'}

os.makedirs('plots', exist_ok=True)
print("Plots will be saved under plots/")


Plots will be saved under plots/


### 11a. Fig. 2 equivalent — Pareto Front Comparison

In [56]:

def pareto_front_from_records(records, algo, d_total):
    """Take the best-per-run (f1, feature_proportion) pairs for one algorithm
    and keep the non-dominated subset. n_feats is a raw COUNT, so it must be
    divided by d_total to get the proportion used on the x-axis."""
    pts = [(1 - r['test_accuracy']/100, r['n_feats'] / d_total) for r in records if r['algo'] == algo]
    pts = sorted(set(pts))
    front = []
    best_f2 = float('inf')
    for f1, f2 in pts:
        if f2 < best_f2:
            front.append((f1, f2)); best_f2 = f2
    return front

FIG2_DATASET, FIG2_BACKEND = 'SpamBase', 'SVM'
records_fig2 = master_results[FIG2_DATASET][FIG2_BACKEND]['records']
d_total_fig2 = master_results[FIG2_DATASET][FIG2_BACKEND]['d_total']

fig, ax = plt.subplots(figsize=(8, 6))
for algo in ['MOGSK', 'NSGA-II', 'MOPSO', 'BPSO']:
    front = pareto_front_from_records(records_fig2, algo, d_total_fig2)
    if not front:
        continue
    xs = [f2 * 100 for _, f2 in front]           # feature proportion (%), now correctly 0-100
    ys = [(1 - f1) * 100 for f1, _ in front]      # accuracy (%)
    marker = 'X' if algo == 'BPSO' else 'o'
    ax.plot(xs, ys, marker=marker, label=algo, color=COLORS[algo], linewidth=2 if algo != 'BPSO' else 0)
ax.set_xlabel('Feature Proportion (%)', fontsize=12)
ax.set_xlim(0, 100)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(f'Pareto Front Comparison — {FIG2_DATASET} ({FIG2_BACKEND})', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig('plots/fig2_pareto_fronts.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved plots/fig2_pareto_fronts.png")


✓ Saved plots/fig2_pareto_fronts.png


### 11b. Fig. 3 equivalent — HV Bar Comparison (primary datasets, SVM backend)

In [57]:

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(DATASETS_PRIMARY))
width = 0.11
for i, algo in enumerate(ALGOS):
    hv_means = [master_results[ds]['SVM']['summary'][algo]['hv_mean'] for ds in DATASETS_PRIMARY]
    hv_stds  = [master_results[ds]['SVM']['summary'][algo]['hv_std']  for ds in DATASETS_PRIMARY]
    ax.bar(x + i*width, hv_means, width, yerr=hv_stds, capsize=3,
           label=algo, color=COLORS[algo], edgecolor='white')
ax.set_xticks(x + width * (len(ALGOS)-1) / 2)
ax.set_xticklabels(DATASETS_PRIMARY)
ax.set_ylabel('Hypervolume (HV)', fontsize=12)
ax.set_title('HV Indicator Comparison — Primary Datasets (SVM Backend)', fontweight='bold')
ax.legend(fontsize=9, ncol=4)
ax.grid(True, axis='y', alpha=0.3)
fig.savefig('plots/fig3_hv_bars.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved plots/fig3_hv_bars.png")


✓ Saved plots/fig3_hv_bars.png


### 11c. Fig. 4 equivalent — Convergence Curves

In [58]:

FIG4_DATASET = 'SpamBase'
fig, ax = plt.subplots(figsize=(9, 6))
for backend in BACKENDS:
    hist = master_results[FIG4_DATASET][backend]['summary']['MOGSK'].get('hv_hist_mean', [])
    if hist:
        ax.plot(range(1, len(hist)+1), hist, linewidth=2.5, label=f'MOGSK ({backend})')
for algo in ['NSGA-II', 'MOPSO', 'MOEA/D']:
    hist = master_results[FIG4_DATASET]['SVM']['summary'][algo].get('hv_hist_mean', [])
    if hist:
        ax.plot(range(1, len(hist)+1), hist, '--', linewidth=1.8, color=COLORS[algo], label=algo)
ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('Hypervolume (HV)', fontsize=12)
ax.set_title(f'Convergence Curves — {FIG4_DATASET}', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.savefig('plots/fig4_convergence.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved plots/fig4_convergence.png")


✓ Saved plots/fig4_convergence.png


### 11d. Fig. 5 equivalent — MOGSK Accuracy Across All 10 Datasets × 3 Backends

In [59]:

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(ALL_DATASETS))
width = 0.25
backend_colors = {'SVM': '#1976D2', 'KNN': '#42A5F5', 'RF': '#81D4FA'}
for i, backend in enumerate(BACKENDS):
    accs = [master_results[ds][backend]['summary']['MOGSK']['acc_mean'] for ds in ALL_DATASETS]
    errs = [master_results[ds][backend]['summary']['MOGSK']['acc_std']  for ds in ALL_DATASETS]
    bars = ax.bar(x + i*width, accs, width, yerr=errs, capsize=3,
                  label=backend, color=backend_colors[backend], edgecolor='white')
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}', ha='center', fontsize=7)
ax.set_xticks(x + width)
ax.set_xticklabels(ALL_DATASETS, rotation=30, ha='right')
ax.set_ylabel('Classification Accuracy (%)', fontsize=12)
ax.set_title('MOGSK Accuracy Across Datasets and Classifier Backends', fontweight='bold')
ax.set_ylim(50, 100)
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
fig.savefig('plots/fig5_accuracy_by_dataset.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved plots/fig5_accuracy_by_dataset.png")


✓ Saved plots/fig5_accuracy_by_dataset.png


### 11e. Fig. 6 equivalent — Feature Count & Reduction % (SVM backend, computed mean)

In [60]:

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: selected vs dropped features
sel = [master_results[ds]['SVM']['summary']['MOGSK']['nfeats_mean'] for ds in ALL_DATASETS]
tot = [master_results[ds]['SVM']['d_total'] for ds in ALL_DATASETS]
dropped = [t - s for t, s in zip(tot, sel)]
axes[0].bar(ALL_DATASETS, sel, label='Selected features', color='#1976D2')
axes[0].bar(ALL_DATASETS, dropped, bottom=sel, label='Dropped features', color='#EF9A9A')
axes[0].set_ylabel('Number of Features'); axes[0].set_title('Feature Count: Selected vs Original', fontweight='bold')
axes[0].legend(); axes[0].tick_params(axis='x', rotation=45)

# Right: reduction % per dataset with COMPUTED mean (addresses R1.5.3)
reduction_pct = [(1 - s/t) * 100 for s, t in zip(sel, tot)]
mean_reduction = np.mean(reduction_pct)
axes[1].barh(ALL_DATASETS, reduction_pct, color='#1976D2')
for i, v in enumerate(reduction_pct):
    axes[1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)
axes[1].axvline(mean_reduction, color='red', linestyle='--', label=f'Mean = {mean_reduction:.1f}%')
axes[1].set_xlabel('Feature Reduction (%)'); axes[1].set_title('Feature Reduction per Dataset (SVM)', fontweight='bold')
axes[1].legend()

fig.tight_layout()
fig.savefig('plots/fig6_feature_reduction.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Saved plots/fig6_feature_reduction.png  |  Computed mean reduction = {mean_reduction:.2f}%")


✓ Saved plots/fig6_feature_reduction.png  |  Computed mean reduction = 50.98%


### 11f. Sensitivity plot (Table 9 companion figure)

In [61]:

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, param in zip(axes, sensitivity_df['Param'].unique()):
    sub = sensitivity_df[sensitivity_df['Param'] == param]
    ax.errorbar(sub['Value'], sub['Acc_mean'], yerr=sub['Acc_std'], marker='o',
                color='#1565C0', linewidth=2, capsize=3)
    ax.set_xlabel(param, fontsize=10)
    ax.set_ylabel('Accuracy (%)', fontsize=10)
    ax.set_title(sub['Label'].iloc[0], fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
fig.suptitle('Sensitivity Analysis — SpamBase (SVM)', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('plots/sensitivity_SpamBase_SVM.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved plots/sensitivity_SpamBase_SVM.png")


✓ Saved plots/sensitivity_SpamBase_SVM.png


## 12. Final Output Manifest

Everything this notebook produces, all generated at run time from the code
above (nothing pre-filled):

In [62]:

print("CSV / JSON tables in results/:")
for f in sorted(os.listdir('results')):
    print("  -", f)

print("\nFigures in plots/:")
for f in sorted(os.listdir('plots')):
    print("  -", f)


CSV / JSON tables in results/:
  - checkpoints
  - hyperparameter_ranges.csv
  - master_results.json
  - runtime_complexity.csv
  - table10_wilcoxon_all8.csv
  - table3_accuracy_SVM.csv
  - table7_hypervolume_KNN.csv
  - table7_hypervolume_RF.csv
  - table7_hypervolume_SVM.csv
  - table8_ablation_all8.csv
  - table9_sensitivity.csv

Figures in plots/:
  - fig2_pareto_fronts.png
  - fig3_hv_bars.png
  - fig4_convergence.png
  - fig5_accuracy_by_dataset.png
  - fig6_feature_reduction.png
  - sensitivity_SpamBase_SVM.png
